In [ ]:
# ============================================================================
# ICTSP ZERO-SHOT LEARNING - COMPLETE 48 EXPERIMENTS
# 6 Directions × 4 Methods × 2 pred_lens (96, 192) = 48 experiments
# Optimized for RTX 2080 Ti (11GB VRAM) | Jupyter Notebook Ready
# ============================================================================

import os
import gc
import json
import time
import random
import warnings
import subprocess
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm  # For Jupyter progress bars

warnings.filterwarnings("ignore")

# ============================================================================
# DEVICE SETUP - AUTO DETECTS RTX 2080 Ti
# ============================================================================
print("=" * 60)
print("GPU DETECTION")
print("=" * 60)

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Detected: {gpu_name}")
    print(f"✅ VRAM: {vram_gb:.1f} GB")
    
    # Optimize for 2080 Ti (11GB)
    if vram_gb < 12:
        print("⚠️ Running on 11GB GPU - using conservative batch sizes")
        BATCH_SIZE_RECOMMENDED = 16
    else:
        BATCH_SIZE_RECOMMENDED = 32
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected - using CPU (will be very slow)")
    BATCH_SIZE_RECOMMENDED = 8

print(f"✅ Using device: {device}")
print(f"✅ Recommended batch size: {BATCH_SIZE_RECOMMENDED}")
print("=" * 60)

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(2024)

# ============================================================================
# OPTIONAL PACKAGES - AUTO INSTALL
# ============================================================================
try:
    import pywt
    HAS_PYWT = True
except ImportError:
    print("Installing PyWavelets...")
    subprocess.check_call(["pip", "install", "-q", "PyWavelets"])
    import pywt
    HAS_PYWT = True

try:
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True
except ImportError:
    print("Installing statsmodels...")
    subprocess.check_call(["pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True

try:
    import gdown
    HAS_GDOWN = True
except ImportError:
    print("Installing gdown...")
    subprocess.check_call(["pip", "install", "-q", "gdown"])
    import gdown
    HAS_GDOWN = True

# ============================================================================
# RECONSTRUCTION METHODS
# ============================================================================

class STLDecomposition:
    @staticmethod
    def get_period(dataset_name: str) -> int:
        if "ETTh" in dataset_name:
            return 24
        elif "ETTm" in dataset_name:
            return 96
        elif "Exchange" in dataset_name:
            return 7
        elif "Weather" in dataset_name:
            return 144
        return 24

    @staticmethod
    def denoise_signal(signal: np.ndarray, period: int, robust: bool = True) -> np.ndarray:
        if not HAS_STL:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8 or len(signal) < 2 * period:
            return signal.astype(np.float32)
        try:
            stl = STLModel(signal, period=period, robust=robust)
            result = stl.fit()
            denoised = result.trend + result.seasonal
            bad = np.isnan(denoised)
            denoised[bad] = signal[bad]
            return denoised.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, dataset_name: str, robust: bool = True) -> np.ndarray:
        period = STLDecomposition.get_period(dataset_name)
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = STLDecomposition.denoise_signal(data[:, c], period, robust)
        return out


class WaveletReconstruction:
    @staticmethod
    def denoise_signal(signal: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        if not HAS_PYWT:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        max_level = max(1, int(np.log2(len(signal))) - 1)
        level = min(level, max_level)
        try:
            coeffs = pywt.wavedec(signal, wavelet, level=level)
            if len(coeffs) > 1:
                sigma = np.median(np.abs(coeffs[-1])) / 0.6745
                threshold = sigma * np.sqrt(2 * np.log(len(signal))) * threshold_factor
                coeffs_thresh = [coeffs[0]]
                for i in range(1, len(coeffs)):
                    coeffs_thresh.append(pywt.threshold(coeffs[i], threshold, mode="soft"))
                out = pywt.waverec(coeffs_thresh, wavelet)[:len(signal)]
            else:
                out = signal
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = WaveletReconstruction.denoise_signal(data[:, c], wavelet, level, threshold_factor)
        return out


class SSARecovery:
    @staticmethod
    def _embed(signal: np.ndarray, L: int) -> np.ndarray:
        N = len(signal)
        K = N - L + 1
        X = np.zeros((L, K), dtype=np.float32)
        for i in range(L):
            X[i, :] = signal[i:i + K]
        return X

    @staticmethod
    def _diagonal_averaging(X: np.ndarray) -> np.ndarray:
        L, K = X.shape
        N = L + K - 1
        signal = np.zeros(N, dtype=np.float32)
        counts = np.zeros(N, dtype=np.float32)
        for i in range(L):
            for j in range(K):
                signal[i + j] += X[i, j]
                counts[i + j] += 1.0
        return signal / np.maximum(counts, 1e-8)

    @staticmethod
    def denoise_signal(signal: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        signal = signal.copy()
        N = len(signal)
        if N < L:
            L = max(2, N // 2)
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        try:
            X = SSARecovery._embed(signal, L)
            U, S, Vt = np.linalg.svd(X, full_matrices=False)
            r = min(r, len(S))
            Xr = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
            out = SSARecovery._diagonal_averaging(Xr)[:N]
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = SSARecovery.denoise_signal(data[:, c], L, r)
        return out


# ============================================================================
# DATA LOADING
# ============================================================================

class DataManager:
    DATASET_URLS = {
        "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
        "ETTh2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv",
        "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
        "ETTm2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv",
        "Exchange": "https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/master/exchange_rate/exchange_rate.txt.gz",
    }
    
    WEATHER_URL = "https://drive.google.com/uc?export=download&id=1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy"
    
    @staticmethod
    def load_weather_from_google_drive() -> np.ndarray:
        print("Downloading Weather dataset...")
        output = "weather.csv"
        if not os.path.exists(output):
            gdown.download(DataManager.WEATHER_URL, output, quiet=False)
        df = pd.read_csv(output)
        if 'date' in df.columns:
            df = df.drop(columns=['date'])
        values = df.values.astype(np.float32)
        print(f"✅ Loaded Weather: {values.shape}")
        return values
    
    @staticmethod
    def load_raw(name: str) -> np.ndarray:
        if name == "Weather":
            return DataManager.load_weather_from_google_drive()
        
        url = DataManager.DATASET_URLS[name]
        print(f"Loading {name}...")
        
        if name == "Exchange":
            import gzip
            from io import BytesIO
            import urllib.request
            with urllib.request.urlopen(url) as response:
                compressed = response.read()
                with gzip.open(BytesIO(compressed), 'rt') as f:
                    # FIXED: Exchange dataset is space-separated, not comma
                    df = pd.read_csv(f, header=None, sep=r'\s+')
        else:
            df = pd.read_csv(url)
            if "date" in df.columns:
                df = df.drop(columns=["date"])
        
        values = df.values.astype(np.float32)
        print(f"✅ Loaded {name}: {values.shape}")
        return values


class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.window_len = seq_len + pred_len
        if len(data) >= self.window_len:
            self.n_windows = len(data) - self.window_len + 1
        else:
            self.n_windows = 0
    
    def __len__(self):
        return self.n_windows
    
    def __getitem__(self, idx):
        window = self.data[idx:idx + self.window_len]
        x_enc = window[:self.seq_len]
        y = window[self.seq_len:self.seq_len + self.pred_len]
        x_mark_dec_dummy = torch.zeros(self.seq_len + self.pred_len, 1)
        return x_enc, y, x_mark_dec_dummy


# ============================================================================
# ICTSP CORE
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.dropout = nn.Dropout(dropout)
        assert self.head_dim * heads == embed_size
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)
    
    def forward(self, values, keys, queries, mask=None):
        N = queries.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], queries.shape[1]
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = queries.reshape(N, query_len, self.heads, self.head_dim)
        values = self.values(values)
        keys = self.dropout(self.keys(keys))
        queries = self.queries(queries)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))
        energy = energy / (self.embed_size ** 0.5)
        attention = F.softmax(energy, dim=-1)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        out = self.fc_out(out)
        return out, attention


class TransformerEncoder(nn.Module):
    def __init__(self, emb_size=128, depth=3, heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=emb_size,
                nhead=heads,
                dim_feedforward=mlp_ratio * emb_size,
                batch_first=True,
                dropout=dropout,
                norm_first=False,
                activation="gelu",
            )
            for _ in range(depth)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Tokenizer(nn.Module):
    def __init__(self, lookback=512, output=96, stride=8):
        super().__init__()
        self.d = lookback + output
        self.s = stride
    
    def forward(self, tensor):
        B, C, L = tensor.shape
        if L >= self.d:
            unfolded = tensor.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)
        else:
            padding = torch.zeros(B, C, self.d - L, device=tensor.device, dtype=tensor.dtype)
            tensor_padded = torch.cat([tensor, padding], dim=-1)
            unfolded = tensor_padded.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)


class ICTSP(nn.Module):
    def __init__(
        self,
        lookback=512,
        output=96,
        depth=3,
        heads=8,
        mlp_ratio=4,
        d_model=128,
        external_stride=8,
        n_channels=7,
        dropout=0.5,
        token_retriever_flag=True,
        token_limit=2048,
    ):
        super().__init__()
        self.lookback = lookback
        self.pred_len = output
        self.external_stride = external_stride
        self.input_projection = nn.Linear(lookback + output, d_model)
        self.transformer_encoder = TransformerEncoder(d_model, depth, heads, mlp_ratio, dropout)
        self.input_norm = nn.LayerNorm(d_model)
        self.output_norm = nn.LayerNorm(d_model)
        self.output_embedding = nn.Parameter(0.01 * torch.randn(1, 1, 1200))
        self.output_projection = nn.Linear(d_model, output)
        self.channel_embedding = nn.Parameter(0.01 * torch.randn(1, 1024, d_model))
        self.pos_embedding = nn.Parameter(0.01 * torch.randn(1, 8192, 1, d_model))
        self.pos_embedding_after = nn.Parameter(0.01 * torch.randn(1, 8192, d_model))
        self.token_retriever_flag = token_retriever_flag
        self.linear_warmup_steps = 5000
        self.linear_warm_up_counter = 0
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, x, x_mark_dec=None):
        lookback = self.lookback
        future = self.pred_len
        mean = x[:, [-1], :].detach()
        x = x.permute(0, 2, 1)
        output_embedding = self.output_embedding[:, :, 0:future].expand(x.shape[0], x.shape[1], -1)
        x = torch.cat([x, output_embedding + mean.permute(0, 2, 1)], dim=-1)
        B, C, _ = x.shape
        number_of_targets = C
        x_orig = x[:, :, 0:-future].clone()
        external_tokenizer = Tokenizer(lookback, future, stride=self.external_stride)
        ex_tokens = external_tokenizer(x_orig)
        _, _, _, d = ex_tokens.shape
        ex_tokens = ex_tokens.permute(0, 2, 1, 3).reshape(B, -1, d)
        x_target = x[:, -number_of_targets:, -(lookback + future):]
        x_tokens = torch.cat([ex_tokens, x_target], dim=1)
        token_mean = x_tokens[:, :, [-(future + 1)]].detach()
        x_tokens = x_tokens - token_mean
        x_tokens = self.input_projection(x_tokens)
        channel_mask = self.channel_embedding[:, -C:, :]
        x_tokens = x_tokens + channel_mask.repeat(1, x_tokens.shape[1] // C, 1)
        pos_emb = self.pos_embedding[:, -(x_tokens.shape[1] // C):, :, :].expand(-1, -1, C, -1)
        pos_emb = pos_emb.reshape(pos_emb.shape[0], pos_emb.shape[1] * pos_emb.shape[2], pos_emb.shape[3])
        x_tokens = x_tokens + pos_emb
        
        if self.linear_warm_up_counter < self.linear_warmup_steps:
            if self.training:
                self.linear_warm_up_counter += 1
            x_output = self.output_projection(x_tokens[:, -number_of_targets:, :])
            x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
            x_output = x_output.permute(0, 2, 1)
            return x_output
        
        x_tokens = self.input_norm(x_tokens)
        x_tokens = self.transformer_encoder(x_tokens)
        x_output = x_tokens[:, -number_of_targets:, :]
        x_output = self.output_norm(x_output)
        x_output = self.output_projection(x_output)
        x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
        x_output = x_output.permute(0, 2, 1)
        return x_output


class RepoICTSPModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = ICTSP(
            lookback=cfg.lookback,
            output=cfg.pred_len,
            depth=cfg.e_layers,
            heads=cfg.n_heads,
            mlp_ratio=cfg.mlp_ratio,
            d_model=cfg.d_model,
            external_stride=cfg.sampling_step,
            n_channels=cfg.enc_in,
            dropout=cfg.dropout,
        )
    
    def forward(self, x, x_mark_dec):
        return self.model(x, x_mark_dec)


# ============================================================================
# BASELINE MODELS
# ============================================================================

class LastValueBaseline(nn.Module):
    def forward(self, history, horizon):
        return history[:, -1:, :].repeat(1, horizon, 1)


class NLinearBaseline(nn.Module):
    def __init__(self, seq_len, horizon, num_channels):
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(seq_len, horizon) for _ in range(num_channels)])
    
    def forward(self, history):
        return torch.stack([self.linears[c](history[:, :, c]) for c in range(history.shape[-1])], dim=2)


# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    dataset_name: str = "ETTh1"
    experiment_mode: str = "zero_shot"
    use_stl: bool = False
    use_wavelet: bool = False
    use_ssa: bool = False
    lookback: int = 512
    pred_len: int = 96
    e_layers: int = 3
    d_model: int = 128
    n_heads: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.5
    sampling_step: int = 8
    batch_size: int = BATCH_SIZE_RECOMMENDED
    batch_size_test: int = BATCH_SIZE_RECOMMENDED
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    train_epochs: int = 100
    patience_steps: int = 20
    grad_clip: float = 1.0
    enc_in: int = 7
    device: str = device
    seed: int = 2024


# ============================================================================
# METRICS & EVALUATION
# ============================================================================

def compute_metrics(pred, target):
    with torch.no_grad():
        mse = F.mse_loss(pred, target).item()
        mae = F.l1_loss(pred, target).item()
    return mse, mae


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_mse = 0.0
    total_mae = 0.0
    n_batches = 0
    
    for x_enc, y, x_mark_dec in loader:
        x_enc = x_enc.to(device)
        y = y.to(device)
        x_mark_dec = x_mark_dec.to(device)
        pred = model(x_enc, x_mark_dec)
        mse, mae = compute_metrics(pred, y)
        total_mse += mse
        total_mae += mae
        n_batches += 1
    
    return {
        "mse": total_mse / n_batches if n_batches > 0 else float("inf"),
        "mae": total_mae / n_batches if n_batches > 0 else float("inf"),
    }


# ============================================================================
# ZERO-SHOT TRAINER - OPTIMIZED FOR 2080 Ti
# ============================================================================

class ZeroShotTrainer:
    def __init__(self, cfg: Config, source_name: str, target_name: str, method_name: str, experiment_id: int, total_exp: int):
        self.cfg = cfg
        self.device = cfg.device
        self.source_name = source_name
        self.target_name = target_name
        self.method_name = method_name
        self.experiment_id = experiment_id
        self.total_exp = total_exp
        set_seed(cfg.seed)
        
        print(f"\n{'='*60}")
        print(f"[{experiment_id}/{total_exp}] {source_name} → {target_name}")
        print(f"Method: {method_name} | pred_len={cfg.pred_len}")
        print(f"{'='*60}")
    
    def apply_reconstruction(self, data: np.ndarray, dataset_name: str) -> np.ndarray:
        if self.cfg.use_stl:
            print(f"  🔧 STL reconstruction (period={STLDecomposition.get_period(dataset_name)})")
            return STLDecomposition.apply_to_multivariate(data, dataset_name, robust=True)
        elif self.cfg.use_wavelet:
            print(f"  🔧 Wavelet reconstruction")
            return WaveletReconstruction.apply_to_multivariate(data, "db4", 2, 0.1)
        elif self.cfg.use_ssa:
            print(f"  🔧 SSA reconstruction")
            return SSARecovery.apply_to_multivariate(data, 30, 5)
        else:
            print(f"  🔧 Baseline (no reconstruction)")
            return data.astype(np.float32)
    
    def prepare_data(self):
        print(f"\n[1/4] Loading data...")
        
        source_values = DataManager.load_raw(self.source_name)
        target_values = DataManager.load_raw(self.target_name)
        
        print(f"  Source {self.source_name}: {source_values.shape}")
        print(f"  Target {self.target_name}: {target_values.shape}")
        
        print(f"\n[2/4] Applying reconstruction...")
        source_reconstructed = self.apply_reconstruction(source_values, self.source_name)
        target_reconstructed = self.apply_reconstruction(target_values, self.target_name)
        
        self.cfg.enc_in = source_reconstructed.shape[1]
        print(f"  Channels: source={self.cfg.enc_in}, target={target_reconstructed.shape[1]}")
        
        # Create splits (70% train, 10% val, 20% unused)
        n_total = len(source_reconstructed)
        n_train = int(n_total * 0.70)
        n_val = int(n_total * 0.10)
        
        train_data = source_reconstructed[:n_train]
        val_data = source_reconstructed[n_train:n_train + n_val]
        
        print(f"\n[3/4] Standardizing using source statistics...")
        self.scaler = StandardScaler()
        self.scaler.fit(train_data)
        
        train_scaled = self.scaler.transform(train_data).astype(np.float32)
        val_scaled = self.scaler.transform(val_data).astype(np.float32)
        target_scaled = self.scaler.transform(target_reconstructed).astype(np.float32)
        
        n_test_start = int(len(target_scaled) * 0.70)
        test_data = target_scaled[n_test_start:]
        
        print(f"  Train samples: {len(train_scaled)}")
        print(f"  Val samples: {len(val_scaled)}")
        print(f"  Test samples: {len(test_data)}")
        
        self.train_ds = WindowDataset(train_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.val_ds = WindowDataset(val_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.test_ds = WindowDataset(test_data, self.cfg.lookback, self.cfg.pred_len)
        
        self.train_loader = DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=True, num_workers=0)
        self.val_loader = DataLoader(self.val_ds, batch_size=self.cfg.batch_size_test, shuffle=False, num_workers=0)
        self.test_loader = DataLoader(self.test_ds, batch_size=self.cfg.batch_size_test, shuffle=False, num_workers=0)
        
        print(f"  Train windows: {len(self.train_ds)}")
        print(f"  Val windows: {len(self.val_ds)}")
        print(f"  Test windows: {len(self.test_ds)}")
    
    def train(self):
        print(f"\n[4/4] Training on {self.source_name}...")
        
        self.model = RepoICTSPModel(self.cfg).to(self.device)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Model parameters: {total_params:,}")
        
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
        
        best_val_mse = float("inf")
        best_state = None
        patience_counter = 0
        
        for epoch in range(1, self.cfg.train_epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0
            
            for x_enc, y, x_mark_dec in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                x_mark_dec = x_mark_dec.to(self.device)
                
                optimizer.zero_grad(set_to_none=True)  # More memory efficient
                pred = self.model(x_enc, x_mark_dec)
                loss = F.mse_loss(pred, y)
                loss.backward()
                
                if self.cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1
            
            val_metrics = evaluate(self.model, self.val_loader, self.device)
            scheduler.step(val_metrics["mse"])
            
            if val_metrics["mse"] < best_val_mse - 1e-8:
                best_val_mse = val_metrics["mse"]
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f} ✓")
            else:
                patience_counter += 1
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f}")
            
            if patience_counter >= self.cfg.patience_steps:
                print(f"  Early stopping at epoch {epoch}")
                break
        
        if best_state is not None:
            self.model.load_state_dict(best_state)
        
        print(f"  Best Val MSE: {best_val_mse:.6f}")
        return best_val_mse
    
    def evaluate_zero_shot(self):
        print(f"\n  📊 Zero-shot evaluation on {self.target_name}...")
        test_metrics = evaluate(self.model, self.test_loader, self.device)
        print(f"  Zero-Shot MSE: {test_metrics['mse']:.6f}")
        print(f"  Zero-Shot MAE: {test_metrics['mae']:.6f}")
        return test_metrics
    
    def evaluate_baselines(self):
        print(f"\n  📊 Baseline evaluation on {self.target_name}...")
        results = {}
        
        # Last Value Baseline
        lv = LastValueBaseline()
        lv_mse = []
        for x_enc, y, _ in self.test_loader:
            x_enc = x_enc.to(self.device)
            y = y.to(self.device)
            pred = lv(x_enc, self.cfg.pred_len)
            mse, _ = compute_metrics(pred, y)
            lv_mse.append(mse)
        results["LastValue"] = float(np.mean(lv_mse))
        
        # NLinear Baseline (quick training)
        nlinear = NLinearBaseline(self.cfg.lookback, self.cfg.pred_len, self.cfg.enc_in).to(self.device)
        opt = torch.optim.Adam(nlinear.parameters(), lr=1e-3)
        
        for _ in range(20):
            nlinear.train()
            for x_enc, y, _ in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                opt.zero_grad(set_to_none=True)
                loss = F.mse_loss(nlinear(x_enc), y)
                loss.backward()
                opt.step()
        
        nlinear.eval()
        nlin_mse = []
        with torch.no_grad():
            for x_enc, y, _ in self.test_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                pred = nlinear(x_enc)
                mse, _ = compute_metrics(pred, y)
                nlin_mse.append(mse)
        results["NLinear"] = float(np.mean(nlin_mse))
        
        print(f"  LastValue MSE: {results['LastValue']:.6f}")
        print(f"  NLinear MSE: {results['NLinear']:.6f}")
        
        return results
    
    def cleanup(self):
        if hasattr(self, "model"):
            del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================================
# MAIN EXPERIMENT RUNNER
# ============================================================================

def run_all_zero_shot_experiments():
    """Run all 48 zero-shot experiments: 6 directions × 4 methods × 2 pred_lens"""
    
    # Zero-shot directions
    zero_shot_directions = [
        ("ETTh1", "ETTh2"),
        ("ETTh2", "ETTh1"),
        ("ETTm1", "ETTm2"),
        ("ETTm2", "ETTm1"),
        ("Exchange", "Weather"),
        ("Weather", "Exchange"),
    ]
    
    # Methods
    methods = [
        ("Baseline", False, False, False),
        ("STL", True, False, False),
        ("Wavelet", False, True, False),
        ("SSA", False, False, True),
    ]
    
    # Prediction lengths
    pred_lengths = [96, 192]
    
    all_results = {}
    
    print("\n" + "🔥" * 50)
    print("ICTSP ZERO-SHOT LEARNING - COMPLETE 48 EXPERIMENTS")
    print(f"Total: 6 directions × 4 methods × 2 pred_lens = 48 experiments")
    print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    print(f"Batch Size: {BATCH_SIZE_RECOMMENDED}")
    print("🔥" * 50)
    
    total_exp = len(zero_shot_directions) * len(methods) * len(pred_lengths)
    exp_count = 0
    
    # Create progress tracking
    start_time = time.time()
    
    for pred_len in pred_lengths:
        for source, target in zero_shot_directions:
            for method_name, use_stl, use_wavelet, use_ssa in methods:
                exp_count += 1
                key = f"{source}_to_{target}_{method_name}_pred{pred_len}"
                
                try:
                    cfg = Config(
                        dataset_name=source,
                        pred_len=pred_len,
                        use_stl=use_stl,
                        use_wavelet=use_wavelet,
                        use_ssa=use_ssa,
                        batch_size=BATCH_SIZE_RECOMMENDED,
                        batch_size_test=BATCH_SIZE_RECOMMENDED,
                    )
                    
                    trainer = ZeroShotTrainer(cfg, source, target, method_name, exp_count, total_exp)
                    trainer.prepare_data()
                    
                    exp_start = time.time()
                    best_val_mse = trainer.train()
                    train_time = time.time() - exp_start
                    
                    test_metrics = trainer.evaluate_zero_shot()
                    baseline_metrics = trainer.evaluate_baselines()
                    
                    all_results[key] = {
                        "source": source,
                        "target": target,
                        "method": method_name,
                        "pred_len": pred_len,
                        "source_channels": cfg.enc_in,
                        "ictsp_zero_shot_mse": test_metrics["mse"],
                        "ictsp_zero_shot_mae": test_metrics["mae"],
                        "best_val_mse": best_val_mse,
                        "lastvalue_mse": baseline_metrics["LastValue"],
                        "nlinear_mse": baseline_metrics["NLinear"],
                        "training_time_seconds": train_time,
                    }
                    
                    trainer.cleanup()
                    
                    # Save progress after each experiment
                    with open("zero_shot_progress_48.json", "w") as f:
                        json.dump(all_results, f, indent=2, default=str)
                    
                    # Show progress
                    elapsed = time.time() - start_time
                    avg_time = elapsed / exp_count
                    remaining = avg_time * (total_exp - exp_count)
                    print(f"\n⏱️  Progress: {exp_count}/{total_exp} | Elapsed: {elapsed/60:.1f} min | Est. remaining: {remaining/60:.1f} min\n")
                    
                except Exception as e:
                    print(f"❌ Error in {key}: {e}")
                    import traceback
                    traceback.print_exc()
                    all_results[key] = {"error": str(e)}
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    
    total_time = time.time() - start_time
    print("\n" + "=" * 60)
    print(f"✅ ALL 48 EXPERIMENTS COMPLETED!")
    print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
    print("=" * 60)
    
    return all_results


# ============================================================================
# RESULTS VISUALIZATION
# ============================================================================

def print_results_summary(results):
    print("\n" + "=" * 100)
    print("ZERO-SHOT RESULTS SUMMARY - 48 EXPERIMENTS")
    print("=" * 100)
    
    results_list = []
    for key, val in results.items():
        if isinstance(val, dict) and "ictsp_zero_shot_mse" in val:
            results_list.append(val)
    
    if results_list:
        df = pd.DataFrame(results_list)
        
        # Pivot tables for each pred_len
        for pred_len in [96, 192]:
            print(f"\n📊 PRED_LEN = {pred_len}")
            print("-" * 80)
            sub_df = df[df['pred_len'] == pred_len]
            pivot = sub_df.pivot_table(
                values='ictsp_zero_shot_mse',
                index=['source', 'target'],
                columns='method',
                aggfunc='first'
            )
            print(pivot.round(6))
        
        # Save to CSV
        df.to_csv("zero_shot_results_48.csv", index=False)
        print(f"\n✅ Saved to zero_shot_results_48.csv")
        
        # Create summary plot
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        directions = [
            ("ETTh1", "ETTh2"),
            ("ETTh2", "ETTh1"),
            ("ETTm1", "ETTm2"),
            ("ETTm2", "ETTm1"),
            ("Exchange", "Weather"),
            ("Weather", "Exchange"),
        ]
        
        methods_list = ['Baseline', 'STL', 'Wavelet', 'SSA']
        
        for idx, (source, target) in enumerate(directions):
            ax = axes[idx]
            sub_df = df[(df['source'] == source) & (df['target'] == target)]
            
            x = np.arange(len(methods_list))
            width = 0.35
            
            for i, pl in enumerate([96, 192]):
                mse_values = []
                for method in methods_list:
                    val = sub_df[(sub_df['method'] == method) & (sub_df['pred_len'] == pl)]['ictsp_zero_shot_mse'].values
                    mse_values.append(val[0] if len(val) > 0 else 0)
                ax.bar(x + i*width, mse_values, width, label=f'pred_len={pl}')
            
            ax.set_title(f'{source} → {target}', fontsize=12)
            ax.set_ylabel('MSE')
            ax.set_xticks(x + width/2)
            ax.set_xticklabels(methods_list, rotation=45)
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig("zero_shot_results_48.png", dpi=150, bbox_inches="tight")
        plt.show()
        
        return df
    else:
        print("No valid results")
        return None


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "🔬" * 50)
    print("ICTSP ZERO-SHOT LEARNING - 48 EXPERIMENTS")
    print("=" * 60)
    print("Directions (6):")
    print("  1. ETTh1 → ETTh2")
    print("  2. ETTh2 → ETTh1")
    print("  3. ETTm1 → ETTm2")
    print("  4. ETTm2 → ETTm1")
    print("  5. Exchange → Weather")
    print("  6. Weather → Exchange")
    print("-" * 60)
    print("Methods (4): Baseline, STL, Wavelet, SSA")
    print("Prediction Lengths (2): 96, 192")
    print(f"Total: 48 experiments")
    print(f"Device: {device}")
    print("🔬" * 50)
    
    # Run all experiments
    results = run_all_zero_shot_experiments()
    
    # Print summary
    df_results = print_results_summary(results)
    
    # Save final results
    with open("zero_shot_all_results_48.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    
    print("\n✅ DONE! Files saved:")
    print("   - zero_shot_results_48.csv")
    print("   - zero_shot_all_results_48.json")
    print("   - zero_shot_results_48.png")

GPU DETECTION
✅ GPU Detected: NVIDIA GeForce RTX 2080 Ti
✅ VRAM: 11.8 GB
⚠️ Running on 11GB GPU - using conservative batch sizes
✅ Using device: cuda
✅ Recommended batch size: 16
Installing gdown...

🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
ICTSP ZERO-SHOT LEARNING - 48 EXPERIMENTS
Directions (6):
  1. ETTh1 → ETTh2
  2. ETTh2 → ETTh1
  3. ETTm1 → ETTm2
  4. ETTm2 → ETTm1
  5. Exchange → Weather
  6. Weather → Exchange
------------------------------------------------------------
Methods (4): Baseline, STL, Wavelet, SSA
Prediction Lengths (2): 96, 192
Total: 48 experiments
Device: cuda
🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
ICTSP ZERO-SHOT LEARNING - COMPLETE 48 EXPERIMENTS
Total: 6 directions × 4 methods × 2 pred_lens = 48 experiments
GPU: NVIDIA GeForce RTX 2080 Ti
Batch Size: 16
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

[1/48] ETTh1 → ETTh2
Method: Baseline | pred_len=96

[1/4] Loading data...
Loading

Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 859, in run_all_zero_shot_experiments
    trainer.prepare_data()
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 625, in prepare_data
    source_values = DataManager.load_raw(self.source_name)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 296, in load_raw
    values = df.values.astype(np.float32)
ValueError: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'


❌ Error in Exchange_to_Weather_STL_pred96: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'

[19/48] Exchange → Weather
Method: Wavelet | pred_len=96

[1/4] Loading data...
Loading Exchange...


Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 859, in run_all_zero_shot_experiments
    trainer.prepare_data()
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 625, in prepare_data
    source_values = DataManager.load_raw(self.source_name)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 296, in load_raw
    values = df.values.astype(np.float32)
ValueError: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'


❌ Error in Exchange_to_Weather_Wavelet_pred96: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'

[20/48] Exchange → Weather
Method: SSA | pred_len=96

[1/4] Loading data...
Loading Exchange...


...

❌ Error in Weather_to_Exchange_Wavelet_pred96: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'

[24/48] Weather → Exchange
Method: SSA | pred_len=96

[1/4] Loading data...
✅ Loaded Weather: (52696, 21)
Loading Exchange...


Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 859, in run_all_zero_shot_experiments
    trainer.prepare_data()
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 626, in prepare_data
    target_values = DataManager.load_raw(self.target_name)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 296, in load_raw
    values = df.values.astype(np.float32)
ValueError: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'


❌ Error in Weather_to_Exchange_SSA_pred96: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'

[25/48] ETTh1 → ETTh2
Method: Baseline | pred_len=192

[1/4] Loading data...
Loading ETTh1...


Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 859, in run_all_zero_shot_experiments
    trainer.prepare_data()
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 626, in prepare_data
    target_values = DataManager.load_raw(self.target_name)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_58260\290386304.py", line 296, in load_raw
    values = df.values.astype(np.float32)
ValueError: could not convert string to float: '0.785500,1.611000,0.861698,0.634196,0.211242,0.006838,0.593000,0.525486'


✅ Loaded ETTh1: (17420, 7)
Loading ETTh2...
✅ Loaded ETTh2: (17420, 7)
  Source ETTh1: (17420, 7)
  Target ETTh2: (17420, 7)

[2/4] Applying reconstruction...
  🔧 Baseline (no reconstruction)
  🔧 Baseline (no reconstruction)
  Channels: source=7, target=7

[3/4] Standardizing using source statistics...
  Train samples: 12194
  Val samples: 1742
  Test samples: 5226
  Train windows: 11491
  Val windows: 1039
  Test windows: 4523

[4/4] Training on ETTh1...
  Model parameters: 2,939,760
  Epoch  10 | Loss 0.379349 | Val MSE 0.412962
